In [ ]:
# models.py - defines the PyTorch architecture for the encoders and the spatial solver

import torch
import torch.nn as nn
import torch.nn.functional as F

class SpatialEncoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, out_channels, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
    def forward(self, x):
        return self.encoder(x)

class InundationSolver(nn.Module):
    def __init__(self):
        super().__init__()
        self.rain_encoder = SpatialEncoder(in_channels=1, out_channels=16) 
        self.terrain_encoder = SpatialEncoder(in_channels=4, out_channels=16) 
        self.state_encoder = SpatialEncoder(in_channels=3, out_channels=16) 
        
        self.fusion_solver = nn.Sequential(
            nn.Conv2d(48, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 3, kernel_size=3, padding=1) 
        )

    def forward(self, rain, terrain, state):
        r_feat = self.rain_encoder(rain)
        t_feat = self.terrain_encoder(terrain)
        s_feat = self.state_encoder(state)
        
        fused_features = torch.cat([r_feat, t_feat, s_feat], dim=1)
        predictions = self.fusion_solver(fused_features)
        
        # Enforce non-negative water depth
        H = F.relu(predictions[:, 0:1, :, :]) 
        U = predictions[:, 1:2, :, :]
        V = predictions[:, 2:3, :, :]
        
        return H, U, V

In [ ]:
# physics_loss.py - mathematical logic for the mass conservation residual to keep it isolated from the neural network weights

import torch

def physics_loss_engine(H_pred, U_pred, V_pred, H_prev, rain, infiltration, dx=30.0, dt=3600.0):
    dH_dt = (H_pred - H_prev) / dt
    
    flux_x = H_pred * U_pred
    flux_y = H_pred * V_pred
    
    d_flux_x = (torch.roll(flux_x, shifts=-1, dims=3) - torch.roll(flux_x, shifts=1, dims=3)) / (2 * dx)
    d_flux_y = (torch.roll(flux_y, shifts=-1, dims=2) - torch.roll(flux_y, shifts=1, dims=2)) / (2 * dx)
    
    R_m_s = (rain / 1000.0) / 3600.0
    I_m_s = (infiltration / 1000.0) / 3600.0
    
    residual = dH_dt + d_flux_x + d_flux_y - (R_m_s - I_m_s)
    mass_conservation_loss = torch.mean(residual ** 2)
    
    return mass_conservation_loss

In [ ]:
# train.py - imports the modules from the other two files, configures hardware acceleration, and runs the training loop

import torch
import torch.nn.functional as F
from models import InundationSolver
from physics_loss import physics_loss_engine

# Automatically target Apple Silicon GPU (Metal Performance Shaders) if available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on device: {device}")

model = InundationSolver().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

lambda_physics = 0.5 
num_epochs = 10

# Dummy Data representing one batch (Batch=8, Channels, Height=256, Width=256)
# Replace these with your actual DataLoader
rain_input = torch.rand((8, 1, 256, 256)).to(device)
terrain_input = torch.rand((8, 4, 256, 256)).to(device)
state_input = torch.rand((8, 3, 256, 256)).to(device)
H_ground_truth = torch.rand((8, 1, 256, 256)).to(device)

model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Forward Pass
    H_pred, U_pred, V_pred = model(rain_input, terrain_input, state_input)
    
    # Extract state variables for physics loss
    H_prev = state_input[:, 0:1, :, :]
    infiltration = state_input[:, 1:2, :, :]
    
    # Supervised Loss
    L_supervised = F.mse_loss(H_pred, H_ground_truth)
    
    # Physics Loss
    L_physics = physics_loss_engine(H_pred, U_pred, V_pred, H_prev, rain_input, infiltration)
    
    # Total Loss
    L_total = L_supervised + (lambda_physics * L_physics)
    
    L_total.backward()
    optimizer.step()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {L_total.item():.4f}")

# Save the trained weights
torch.save(model.state_dict(), "inundation_solver_weights.pth")
print("Model saved to inundation_solver_weights.pth")